# 00 - Setup check

**What you will take away**
1. A Python environment where `qiskit`, `qiskit-ibm-runtime` and `qiskit-aer` import and print the versions we expect.
2. A saved IBM Quantum Platform account (Open Plan) and a list of the QPUs you can reach.
3. A two-qubit Bell state you ran yourself on the local simulator with `SamplerV2`.

Ten minutes. If something breaks here, fix it here - everything later in the workshop builds on this notebook.

Sources: [Set up your IBM Cloud account](https://quantum.cloud.ibm.com/docs/guides/cloud-setup),
[Save your login credentials](https://quantum.cloud.ibm.com/docs/guides/save-credentials),
[Initialize your account](https://quantum.cloud.ibm.com/docs/guides/initialize-account).

## 1. Install (only if you have not already)

Uncomment and run once. If you installed from the pre-work email, skip it.

In [ ]:
# %pip install 'qiskit[visualization]' qiskit-ibm-runtime qiskit-aer pylatexenc

## 2. Versions

Everything in this workshop is written for **Qiskit 2.x** and **qiskit-ibm-runtime 0.4x**.
If you see Qiskit 0.x or 1.x, stop and upgrade - the old `execute()` / `IBMQ` / V1-primitive code
you may find online (or get from a chatbot) does not run here. We come back to that in the red-team block.

In [ ]:
import qiskit, qiskit_ibm_runtime, qiskit_aer

print("qiskit            :", qiskit.__version__)
print("qiskit-ibm-runtime:", qiskit_ibm_runtime.__version__)
print("qiskit-aer        :", qiskit_aer.__version__)

assert qiskit.__version__.startswith("2."), "Please upgrade to Qiskit 2.x"
print("\nVersions OK.")

## 3. Your IBM Quantum Platform account

> **Run the next two cells only if you have an account.** Set `HAVE_ACCOUNT = True` first.

Get the two pieces of information from [IBM Quantum Platform](https://quantum.cloud.ibm.com/):

1. In the header, select the **us-east** region (the Open Plan lives there; you only see instances of the region you are logged into).
2. Create an **instance** on the *Open* plan, and copy its **CRN**.
3. From the dashboard, create your **API key** (44 characters) and copy it somewhere safe - it is shown once.

`save_account()` writes the credentials to `~/.qiskit/qiskit-ibm.json`, so you only do this once per machine.
The default channel is `ibm_quantum_platform`, so you no longer have to pass `channel=`.
Treat the API key like a password: do not paste it into a notebook you will share.

In [ ]:
HAVE_ACCOUNT = False          # <- set to True once you have your API key and CRN

from qiskit_ibm_runtime import QiskitRuntimeService

if HAVE_ACCOUNT:
    QiskitRuntimeService.save_account(
        token="<your-44-character-api-key>",
        instance="<your Open Plan CRN or instance name>",
        name="open",           # optional: a label, so you can load it by name later
        set_as_default=True,
        overwrite=True,
    )
    print("Credentials saved.")
else:
    print("Skipped: set HAVE_ACCOUNT = True to save your credentials.")

## 4. What can you reach?

`QiskitRuntimeService()` loads the credentials you just saved. `service.backends()` lists the QPUs your
instance can run on, and `least_busy()` picks the one with the shortest queue - that is the one we use for
the hardware moment later.

Open Plan reminder: about **10 minutes of QPU time per rolling 28 days**, job and Batch mode only
(no Sessions), Heron QPUs in us-east. Every job in this workshop costs seconds, not minutes.

In [ ]:
if HAVE_ACCOUNT:
    service = QiskitRuntimeService()          # or QiskitRuntimeService(name="open")
    for b in service.backends():
        print(f"{b.name:20s} qubits={b.num_qubits:4d}  queue={b.status().pending_jobs}")
    print("\nleast busy:", service.least_busy(operational=True, simulator=False).name)
else:
    print("Skipped: no account configured in this notebook.")

## 5. A Bell state, locally

No account needed for this one. We build the circuit, compile it for the target backend
(here the local `AerSimulator`), and sample it with **SamplerV2** - the same four-step shape
(Map, Optimize, Execute, Post-process) you will see on real hardware.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_histogram

# Map: a two-qubit entangled state
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Optimize: compile to the backend's instruction set
backend = AerSimulator()
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
isa_qc = pm.run(qc)

# Execute
job = Sampler(mode=backend).run([isa_qc], shots=1024)
counts = job.result()[0].data.meas.get_counts()

# Post-process
print(counts)
plot_histogram(counts)

## 6. You are ready when...

- [ ] `qiskit` prints a **2.x** version.
- [ ] `qiskit_ibm_runtime` prints **0.4x**, `qiskit_aer` prints **0.17.x**.
- [ ] The Bell histogram shows only `00` and `11`, roughly 50/50 (no `01` or `10` - it is a simulator, there is no noise yet).
- [ ] *(with an account)* `service.backends()` printed at least one Heron QPU.
- [ ] Your API key is **not** typed into any notebook you plan to share.

Stuck? The two usual causes: an old environment (Qiskit 1.x) or the wrong region
(the Open Plan instance is only visible in **us-east**).

## Key takeaway

Qiskit 2.x has one execution path: **build a circuit -> compile it for a specific backend -> run it through a
primitive (`SamplerV2` / `EstimatorV2`) -> read the counts**. Anything that starts with `execute(...)`,
`IBMQ.load_account()` or `Aer.get_backend(...)` is pre-1.0 code and will not run today.
The local simulator and a real QPU take exactly the same four steps - only the backend object changes.